# Tables in pandas

*Loading, filtering, grouping, joining, and exporting data with pandas*

In [Chapter 2](https://pyba.murtaza.cc/parts/part-01-foundations/ch-02-python-essentials.html) we represented a table as a list of dictionaries. In this chapter we adopt the standard Python tool for tables: **pandas**, the library analytics practitioners use to load, clean, reshape, and summarize tables. It is the single most-used tool in this book. 

In this chapter we also introduce Prairie Wholesale's data: three years of orders, the product catalog, and the customer list. By the end you will have used the data to answer business questions. Which channel generates the most revenue? Which product categories have the best margins? Finally, you will have combined the three raw files into one clean table and exported it. Such prepared tables are the input to other tools: for example, the dashboard we build in [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html) loads one.

> **Setup for this chapter**
>
> You can run this chapter's notebook in two ways. In the cloud, [**open ch-03-pandas.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-03-pandas.ipynb); nothing needs to be installed. Locally, use the `pyba-core` environment (Appendix A). Either way, run the setup cell below first. No keys or paid accounts are needed anywhere in this book; Colab requires only a free Google account.

In [ ]:
# Setup. Run this cell once per session. It installs this chapter's
# packages; on Google Colab it also fetches the course data.
%pip install -q pandas openpyxl
import sys
if "google.colab" in sys.modules:
    !test -d pyba-companion || git clone --quiet --depth 1 https://github.com/murtaza-nasir/pyba-companion.git
    sys.path.insert(0, "pyba-companion")   # makes `import pyba` (DATA_DIR) work

> **Tools in this chapter**
>
> | Tool | Why we use it here | Alternatives | Trade-off |
> |---|---|---|---|
> | pandas | The standard Python library for tables: loading, filtering, grouping, joining | Polars (faster, newer), raw SQL | pandas has the largest ecosystem and the most documentation; widely used in industry |
> | openpyxl | Enables pandas to write `.xlsx` files for handoffs to colleagues | csv only | Excel files carry types and multiple sheets; CSV is simpler and universal |
>
> : {tbl-colwidths="[12,33,22,33]"}

## Approaching a new dataset {#sec-ch3-intro}

Prairie Wholesale's data comes as four CSV files in the companion repository, and every column of every file is documented in Appendix C. One name in the loading cell below is not from pandas: `DATA_DIR`, a path constant from the `pyba` package. It points to the repository's data folder, so the same notebook runs unchanged on any machine and on Colab. Everything else in this chapter is plain pandas.

In [ ]:
import pandas as pd

from pyba import DATA_DIR   # DATA_DIR is the absolute path to the course data folder

orders = pd.read_csv(DATA_DIR / "pw_orders.csv", parse_dates=["order_date"])
products = pd.read_csv(DATA_DIR / "pw_products.csv")
customers = pd.read_csv(DATA_DIR / "pw_customers.csv", parse_dates=["signup_date"])

`read_csv` returns a **DataFrame**: a table with named columns, in which every column has a defined data type from [Chapter 2](https://pyba.murtaza.cc/parts/part-01-foundations/ch-02-python-essentials.html), such as `int`, `float`, or `str`. With the `parse_dates` argument, we specify which columns contain dates. Without it, dates are loaded as plain text. The type matters for the questions we can ask: for two dates, we can ask which came earlier; for two pieces of text, comparison is alphabetical, and alphabetical order matches date order only by accident. This is the first of many small type checks that prevent silent failures later.

Three commands provide a first look at any new table. `head` shows the first rows. `shape` returns (rows, columns). `info` lists every column with its type and how many missing values it contains.

In [ ]:
orders.head()

In [ ]:
orders.shape

In [ ]:
orders.info()

Each row is one **order line**: one product in one order. Order `O000001` may have five rows, one for each SKU. The columns define one line: which order it is part of, when, for which customer, through which channel, with what product, how many units, at which price, with which discount. `line_total` is the line's revenue, `quantity * unit_price * (1 - discount_pct)`, rounded to the cent.

The other two tables are small. `products` has one row per SKU, its category, cost, price, and shelf volume. `customers` has one row per account, its business type, city, region, and signup date.

In [ ]:
products.head(3)

## Columns are Series

A single column, selected with square brackets, is a **Series**. Most column operations are whole-column at once. You will notice that this chapter contains no loops over a column's values. As we noted in [Chapter 2](https://pyba.murtaza.cc/parts/part-01-foundations/ch-02-python-essentials.html), most column operations are vectorized and run much faster than an explicit loop.

In [ ]:
orders["line_total"].sum().round(2)   # total revenue, all three years

In [ ]:
orders["line_total"].describe().round(2)

`describe` collapses a column into its count, mean, spread, and quartiles. Note the mean, the 50% row, and the max. The mean line total is roughly \$131, the median is about \$48, and the max is over \$10,000. The gap between the mean and the median signals something about the data, and we examine such gaps throughout [Chapter 4](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-04-single-variable.html). For now, note how little code the overview required.

Arithmetic on Series applies to every value in a column at once. The cell below computes two new Series, each holding one value per order line. The first, `list_value`, is quantity times unit price: the line's value at full price, before any discount. The second, `discount_dollars`, is that full-price value minus the actual `line_total`: the dollars given up to the discount on that line. The printed number is the sum of the second Series, the total cost of discounts over the three years:

In [ ]:
list_value = orders["quantity"] * orders["unit_price"]
discount_dollars = list_value - orders["line_total"]
discount_dollars.sum().round(2)      # what discounts cost over three years

Date columns have a special accessor, `.dt`, with the calendar parts as attributes:

In [ ]:
orders["order_date"].dt.year.value_counts().sort_index()

`value_counts` counts how many times each distinct value appears in a column. We use it as a quick sanity check on any new table. The counts for 2023 and 2026 are lower because the data covers only half of each year. The counts for the two full years, 2024 and 2025, are around 46,500 and 50,300 lines.

## Filtering rows

For most analyses, we do not need the whole table. We need a subset: for example, the online orders, the orders over \$200, or the August school rush. In pandas, rows are selected with a **boolean mask**: a true/false Series used as a filter.

In [ ]:
online = orders[orders["channel"] == "online"]
online.shape

Conditions are combined with `&` (and) and `|` (or), with each condition requiring its own parentheses:

In [ ]:
big_online = orders[(orders["channel"] == "online") & (orders["line_total"] > 200)]
len(big_online)

In [ ]:
aug_2025 = orders[(orders["order_date"] >= "2025-08-01") & (orders["order_date"] < "2025-09-01")]
aug_2025["line_total"].sum().round(2)

`sort_values` sorts the rows. Followed by `head`, it produces any "top N" listing:

In [ ]:
orders.sort_values("line_total", ascending=False).head(3)

> **Chained assignment**
>
> In the cells above, we saved each filtered result under a new variable name. We recommend this practice. A common mistake is to filter and assign in a single step, as in `orders[orders.channel == "online"]["flag"] = 1`. This form is called **chained assignment**, because two selections are chained together before the assignment. The code appears to add a flag to the online orders in `orders`. In reality, the filter returns a copy of those rows, the value 1 is written into the copy, and the copy is then discarded. The original table is unchanged. Recent versions of pandas detect this mistake and report a `ChainedAssignmentError` warning. When rows of the original table must be updated directly, the correct form is a single selection with `.loc`: for example, `orders.loc[orders.channel == "online", "flag"] = 1`.

## New columns

To create a new column, we assign values to a column name that does not exist yet, and pandas adds the column to the table. The naming habit from [Chapter 2](https://pyba.murtaza.cc/parts/part-01-foundations/ch-02-python-essentials.html) applies to columns as well: name the column according to what it contains. We add one new habit here: whenever possible, calculate a new column from existing columns. A calculated column can be recomputed and checked at any time, and if a source column is ever corrected, the calculated column can be rebuilt from it.

In [ ]:
orders["list_value"] = orders["quantity"] * orders["unit_price"]
orders["discount_dollars"] = orders["list_value"] - orders["line_total"]
orders[["order_id", "sku", "list_value", "discount_dollars", "line_total"]].head(3)

Note the double brackets on the last line. The inner pair is a Python list holding the column names we want, in order. The outer pair selects those columns from the table, exactly as a single column was selected earlier.

> **Python note: method chaining**
>
> pandas methods return a new DataFrame, so calls can be chained: `orders.sort_values(...).head(3)`. A long chain is most readable wrapped in parentheses, one method per line. You will see this layout throughout the book:
>
> ```python
> top_channels = (
>     orders
>     .groupby("channel")["line_total"]
>     .sum()
>     .sort_values(ascending=False)
> )
> ```
>
> With the parentheses, the chain can span lines without backslashes. Read it top to bottom, like a recipe.

## Grouping and aggregating

Many business questions are answered with the same sequence of table operations: split the rows into groups, calculate a summary for each group, and compare the groups. For example, revenue *by channel*, orders *by region*, and margin *by category* all follow this sequence. In pandas, the sequence takes the form of `groupby`.

In [ ]:
orders.groupby("channel")["line_total"].sum().round(2)

Read it as a sentence: split the order lines by channel, take each group's `line_total` column, and sum it. The rep channel accounts for most of the revenue. The online channel is the smallest of the three.

With `agg`, we calculate several summaries at once and choose a name for each one:

In [ ]:
by_channel = orders.groupby("channel").agg(
    revenue=("line_total", "sum"),
    lines=("order_id", "size"),
    avg_discount=("discount_pct", "mean"),
)
by_channel.round(3)

Each argument to `agg` has the form `new_name=(column, function)`: the name for the result, the column to summarize, and the summary function to apply. The result is itself a DataFrame, so you can sort, filter, and export this new table like any other.

## Joining tables

The revenue-by-channel table used only `orders`. Now consider a question that cannot be answered from the order table alone: which *product categories* have the best margins? Category is a fact about the product, stored in the product table. The link between the two tables is the `sku` column they have in common. We can use this link with `merge` to join the tables:

In [ ]:
product_cols = products[["sku", "product_name", "category", "unit_cost"]]
lines = orders.merge(product_cols, on="sku", how="left", validate="many_to_one")
lines[["order_id", "sku", "product_name", "category", "line_total"]].head(3)

We make one choice before the join. Both tables have a `unit_price` column, and joining both copies would produce two confusingly named columns, `unit_price_x` and `unit_price_y`. We therefore select only the product columns we need before merging. Three arguments of `merge` deserve attention. With `on="sku"`, we specify the column that links the two tables. With `how="left"`, we keep every order line in the result, whether or not its SKU matches a product. With `validate="many_to_one"`, we ask pandas to check the join's structure: many order lines may share one product, but no SKU may appear more than once in `products`. If a SKU is repeated, the check stops the join with an error. Without the check, a repeated SKU silently duplicates the matching order lines in the result. Every total calculated from such a table will be inaccurately inflated.

::: {.content-visible when-format="html"}
The explorer below demonstrates that failure on a small example. Repeat a product row, join again, and read the revenue total. Then turn on `validate` and repeat.

<iframe src="../../assets/demos/join-inflation.html" width="100%" height="420" style="border:1px solid #d0d7de; border-radius:8px;" title="join inflation"></iframe>
:::

::: {.content-visible unless-format="html"}
The online edition has an explorer here: a four-line order table joined to a product table, where repeating one product row silently inflates the joined revenue, and `validate="many_to_one"` turns the same mistake into an immediate error.
:::


After the join, each order line has both its revenue and its cost. We calculate the margin on each line by subtracting the line's total cost from its revenue, and we summarize margin by category with a `groupby` of the same form as the ones above:

In [ ]:
lines["margin_dollars"] = lines["line_total"] - lines["quantity"] * lines["unit_cost"]

by_category = lines.groupby("category").agg(
    revenue=("line_total", "sum"),
    margin=("margin_dollars", "sum"),
)
by_category["margin_pct"] = (by_category["margin"] / by_category["revenue"]).round(3)
by_category.sort_values("margin_pct", ascending=False).round(2)

A second join adds the customer's region and business type, so the three tables are now combined into one:

In [ ]:
lines = lines.merge(customers[["customer_id", "region", "business_type"]],
                    on="customer_id", how="left", validate="many_to_one")

## Pivot tables: two groupings at once

`groupby` with two keys produces a long table: one row for every combination of the two keys. In a **pivot table**, the second key's values become the columns instead:

In [ ]:
pivot = lines.pivot_table(
    values="line_total", index="region", columns="category",
    aggfunc="sum",
)
pivot.round(0)

The long form, with one row per region and category pair, is called **tidy data**: every variable is a column, and every observation is a row. The operations of this chapter, from filtering to `groupby`, all expect the tidy form. The wide form is easier for a person to read. We switch between the two layouts with `pivot_table` and its inverse, `melt`. Our rule in this book is to compute in tidy form and to pivot only for presentation or export.

## Exporting prepared data

We share prepared tables with colleagues and with other software: for example, with a colleague who works in Excel, with a BI tool, or with a dashboard such as the one we build in [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html). Two export methods cover these needs. `to_csv` writes a CSV file, and nearly every tool can read the CSV format. `to_excel` writes an Excel file. Excel files preserve the column types, and they are more convenient for a colleague who opens the file directly. To write Excel files, pandas uses **openpyxl**, a small open-source library.

In [ ]:
handoff = lines[["order_id", "order_date", "customer_id", "region", "business_type",
                 "channel", "sku", "product_name", "category", "quantity",
                 "discount_pct", "line_total", "margin_dollars"]]
EXPORTS = DATA_DIR / "cache"          # generated files stay out of the raw data
EXPORTS.mkdir(parents=True, exist_ok=True)
handoff.to_csv(EXPORTS / "pw_lines_prepared.csv", index=False)
by_category.to_excel(EXPORTS / "category_summary.xlsx")   # small table, Excel format
len(handoff)

`index=False` drops the row numbers, which are meaningless outside of pandas. The exported table is intentionally clean and line-level. The tools that consume the file will aggregate it themselves, so we export it unaggregated.

## Evaluation: does the prepared table reconcile?

In this chapter we constructed a joined, enriched, line-level table. Before we rely on the table, one question must be answered: did the joins preserve the data, or did they silently drop or duplicate rows?

|  |  |
|---|---|
| **Metric** | row counts and total revenue, before and after the pipeline. |
| **Test** | the raw `orders` table is the baseline, and `handoff` is the variant. |
| **Standard** | same row counts, and revenue exactly the same to the cent. |

: {tbl-colwidths="[18,82]"}

In [ ]:
checks = {
    "rows_raw": len(orders),
    "rows_handoff": len(handoff),
    "revenue_raw": orders["line_total"].sum().round(2),
    "revenue_handoff": handoff["line_total"].sum().round(2),
    "null_categories": int(handoff["category"].isna().sum()),
    "null_regions": int(handoff["region"].isna().sum()),
}
checks

In [ ]:
assert checks["rows_raw"] == checks["rows_handoff"]
assert checks["revenue_raw"] == checks["revenue_handoff"]
assert checks["null_categories"] == 0 and checks["null_regions"] == 0
print("Reconciled: joins preserved every row and every dollar.")

We follow this reconciliation practice throughout the book. We give every join a `validate` argument, and we end every data-preparation pipeline by comparing a computed total against a number the company already tracks, such as total revenue. For example, the dashboard in [Chapter 6](https://pyba.murtaza.cc/parts/part-02-descriptive/ch-06-dashboards.html) reports total revenue from the table we exported above. Because of the checks in this section, that revenue figure matches the raw order data.

## The decision this informs

Every fall, Prairie Wholesale's managers choose one product category for the fall promotion. The category table above informs this choice. Safety products have the highest margin percentage, at around 36%. Janitorial products have the highest total revenue. A promotion for janitorial protects the company's largest revenue stream. A promotion for safety grows its most profitable one. In [Chapter 13](https://pyba.murtaza.cc/parts/part-05-widening/ch-13-optimization.html) we turn this tradeoff into an optimization.

## Exercises



### Build lab

Build the top-ten customer table for 2025. Filter `lines` to orders placed in 2025. Group by `customer_id`, and compute three summaries: total revenue, number of distinct orders (`("order_id", "nunique")`), and average discount. The grouped table holds `customer_id` and those three columns and nothing else, since `groupby(...).agg(...)` keeps only the grouping key and the columns you name, so join the customer's name and business type back on from `customers` (select `customer_id`, `name`, and `business_type` before merging). Sort by revenue and keep the top ten. Which business types dominate the list?

### Evaluate lab

Reconcile your top-ten table. First, `assert` that the revenue column of your full 2025 grouping (before taking the top ten) sums to the same value as `lines` filtered to 2025, to the cent. Second, verify the join added no rows: the grouped table must have exactly one row per customer before and after joining the names. State in one sentence what a failure of each check would have meant.

> **Lab starter**
>
> A starter notebook for this lab is provided. It restates the task, reproduces the objects from this chapter that the lab builds on, and marks the cells you complete. [**Open ch-03-lab-starter.ipynb in Google Colab**](https://colab.research.google.com/github/murtaza-nasir/pyba-companion/blob/main/notebooks/ch-03-lab-starter.ipynb), or download it from the course page in Blackboard. Colab opens a notebook from GitHub read-only: click **Copy to Drive** in the toolbar before you edit anything, and work in that copy.